# Notebook 07 — Train on BS-80K, test on Machine 1 and Machine 2 separately (GPU, resume-safe)
**Needs:** notebook 01 (`bs80k/bs80k_prepared.npz`, `bs80k/split.csv`), notebook 02 (`prepared_v3`) and notebook 03 (`revision/machine_v3`).

Aim: *train on BS-80K, test on Machine 1 and Machine 2 separately, binary classification only (BS-80K is binary),
and check which method gives the best cross-dataset generalisation.*

**Design (fixed before running):**
* Source: BS-80K (metastasis vs normal), the same train / validation / test split as notebook 01 (2,597 / 325 / 325 patients).
  Early stopping on the BS-80K validation split only; the BS-80K test split is used as a source-domain check.
* Targets: all classification patients of Machine 1 (166) and Machine 2 (178), never used for training. The model gives the
  probability of metastasis, which is evaluated on three binary tasks: **T3 metastatic vs normal** (same classes as BS-80K, primary),
  T2 metastatic vs non-metastatic and T4 metastatic vs degenerative.
* Methods (standard settings, not tuned, same code as Steps 16 and 17):
  * **none** — trained on BS-80K only.
  * **recal** — the 'none' model with its decision threshold set on 20 labelled target patients (class-stratified, from the other
    four folds; 20 repetitions; no training).
  * **iaug** — intensity augmentation; **mixstyle** — MixStyle (trained on BS-80K only).
  * **dann**, **coralstd** — BS-80K labelled + unlabelled images of the target machine. The unlabelled images come from the
    target machine's *primary* (indeterminate) patients (Machine 1: 25, Machine 2: 60), who are never tested, so no test patient is
    seen in training, not even without its label. One model per target machine.
* 4 backbones, ImageNet start, 3 seeds. Runs per seed: none 4 + iaug 4 + mixstyle 4 + dann 8 + coralstd 8 = 28; 84 runs in total.
  Seed 42 first. Epochs 20, patience 5, batch 16 (same as notebook 01).

**Analysis (last cells, CPU, works on partial results):** per target machine and task — every method vs 'none' with a joint patient
bootstrap (10,000), Holm correction; references: the target machine's own model (Experiment 1) and the other machine's model (Experiment 2).

**Runtime:** GPU → Run all. First `QUICK_TEST = True`, then `False`. Safe to re-run: finished runs are skipped.

In [ ]:
# settings
import os, sys, subprocess, time, copy, random, json, gc
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'huggingface_hub', 'python-docx'], check=False)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import timm
from PIL import Image
from sklearn.model_selection import train_test_split

LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
PREP, CV_IN, BSD = BASE + '/prepared', BASE + '/cv5_results', BASE + '/bs80k'
EXP12 = BASE + '/revision/machine_v3'
OUT = BASE + '/revision/bs80k_to_machines'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NOT FOUND - select Runtime > Change runtime type > GPU')

QUICK_TEST = True            # first True (short check), then False
MAX_HOURS = 23.0             # stop after this many hours; run all again to resume (finished runs are skipped)
RUN_TRAINING = True          # set False to re-run only the analysis

MODELS = ['tf_efficientnetv2_s', 'resnext50_32x4d', 'regnety_016', 'inception_v3']
MN = {'tf_efficientnetv2_s': 'EfficientNetV2-S', 'resnext50_32x4d': 'ResNeXt50', 'regnety_016': 'RegNetY-1.6GF', 'inception_v3': 'Inception-v3'}
TASKS = {'T3': ['SECONDARY', 'NORMAL'], 'T2': ['SECONDARY', 'DEGENERATIVE', 'NORMAL'], 'T4': ['SECONDARY', 'DEGENERATIVE']}
E12_TASK = {'T3': 'sec_vs_normal', 'T2': 'cancer_vs_noncancer', 'T4': 'sec_vs_degen'}
SEEDS = [42, 43, 44]
METHODS = ['none', 'iaug', 'mixstyle', 'dann', 'coralstd']
DA_METHODS = ['dann', 'coralstd']
CAMERAS = ['M1', 'M2']
DA_WEIGHT = {'dann': 1.0, 'coral': 1.0, 'coralstd': 1.0}
N_RECAL, RECAL_REPS = 20, 20
N_BOOT = 10000

N_FOLDS, EPOCHS, PATIENCE, BATCH, H, W = 5, 20, 5, 16, 512, 176
INNER_FRAC = 0.15
if LOCAL_TEST:
    BATCH = 4
RAD_REPO, RAD_FILE = 'Lab-Rasool/RadImageNet', 'InceptionV3.pt'     # not used here (shared code)
PRETRAINED = not LOCAL_TEST

if QUICK_TEST:
    MODELS, SEEDS, EPOCHS, N_BOOT = ['regnety_016'], [42], 1, 200
    OUT = BASE + '/revision/bs80k_to_machines_quick'
QUICK_FOLDS = list(range(N_FOLDS))
for d in ['preds', 'logs', 'analysis']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
RUNS = OUT + '/runs.csv'
T_START = time.time()
print('OUT:', OUT, '| QUICK_TEST:', QUICK_TEST)

In [ ]:
# data: our two machines (prepared_v3) + BS-80K (notebook 01 cache), all in one GPU tensor
P3 = BASE + '/prepared_v3'
fix = lambda p: P3 + '/images/' + os.path.basename(str(p))
allp = pd.read_csv(P3 + '/patients.csv')
sp = allp[allp.split == 'pool'][['patient_id', 'machine', 'label', 'ANT', 'POST', 'fold']].reset_index(drop=True)
sp['fold'] = sp.fold.astype(int)
ind = allp[allp.split == 'ind'][['patient_id', 'machine', 'ANT', 'POST']].reset_index(drop=True)
assert sp.patient_id.is_unique and sp.label.isin(['SECONDARY', 'DEGENERATIVE', 'NORMAL']).all()
print('classification patients:', len(sp), '| primary (unlabelled pool for DANN/CORAL):', ind.machine.value_counts().to_dict())

bsplit = pd.read_csv(BSD + '/split.csv')[['patient_id', 'y', 'split']]
bsplit['patient_id'] = 'bs_' + bsplit.patient_id.astype(str)
z = np.load(BSD + '/bs80k_prepared.npz', allow_pickle=True)
zid = {'bs_' + str(p): i for i, p in enumerate(z['ids'])}
assert bsplit.patient_id.isin(zid).all(), 'BS-80K split and image cache do not match'
if QUICK_TEST:
    bsplit = bsplit[bsplit.groupby(['split', 'y']).cumcount() < (10 if LOCAL_TEST else 40)].reset_index(drop=True)
print('BS-80K:', pd.crosstab(bsplit.y.map({0: 'normal', 1: 'metastasis'}), bsplit.split).to_dict())

def load_img(p):
    return np.array(Image.open(fix(p)).convert('L'))
ours = pd.concat([sp[['patient_id', 'machine', 'ANT', 'POST']], ind])
X_ours = np.stack([np.stack([load_img(a), load_img(b)]) for a, b in zip(ours.ANT, ours.POST)])
X_bs = np.stack([np.stack([z['ant'][zid[p]], z['post'][zid[p]]]) for p in bsplit.patient_id])
assert X_ours.shape[2:] == (H, W) and X_bs.shape[2:] == (H, W), (X_ours.shape, X_bs.shape)
X = torch.from_numpy(np.concatenate([X_ours, X_bs])).to(DEVICE)
ALL_IDS = list(ours.patient_id) + list(bsplit.patient_id)
IDX = {p: i for i, p in enumerate(ALL_IDS)}
CAM = {**dict(zip(ours.patient_id, ours.machine)), **{p: 'BS' for p in bsplit.patient_id}}
assert len(IDX) == len(ALL_IDS), 'duplicate patient ids'
del z, X_ours, X_bs
print('images on', DEVICE, tuple(X.shape), f'{X.numel() / 1e6:.0f} MB')

In [ ]:
# model, GPU augmentation, training
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def augment(img, rng):
    """Same augmentation as the paper (RandomAffine ±7°, ±4 %, 0.93–1.07, white fill; brightness/contrast ±20 %), done on the GPU per image."""
    out = []
    for x in img:                                                   # x: 1 x H x W in [0,1]
        ang = float(rng.uniform(-7, 7)); sc = float(rng.uniform(0.93, 1.07))
        tx, ty = int(round(rng.uniform(-0.04, 0.04) * W)), int(round(rng.uniform(-0.04, 0.04) * H))
        x = TF.affine(x, angle=ang, translate=[tx, ty], scale=sc, shear=[0.0], interpolation=InterpolationMode.NEAREST, fill=[1.0])
        ops = [('b', float(rng.uniform(0.8, 1.2))), ('c', float(rng.uniform(0.8, 1.2)))]
        if rng.random() < 0.5:
            ops = ops[::-1]
        for k, f in ops:
            x = (x * f).clamp(0, 1) if k == 'b' else ((x - x.mean()) * f + x.mean()).clamp(0, 1)
        out.append(x)
    return torch.stack(out)

def to_input(idx, train, rng=None):
    x = X[idx].float() / 255.0                                       # B x 2 x H x W
    a, p = x[:, :1], x[:, 1:]
    if train:
        a, p = augment(a, rng), augment(p, rng)
    norm = lambda t: (t.repeat(1, 3, 1, 1) - MEAN) / STD
    return norm(a), norm(p)

class Pooled(nn.Module):
    """Wraps any feature extractor (e.g. RadImageNet full-model object) so that it returns B x C."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        y = self.m(x)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if hasattr(y, 'logits'):
            y = y.logits
        return F.adaptive_avg_pool2d(y, 1).flatten(1) if y.dim() == 4 else y.flatten(1)

_RAD_CACHE = {}
def radimagenet_backbone():
    if 'm' not in _RAD_CACHE:
        if LOCAL_TEST:
            path = BASE + '/radimagenet_test.pt'
        else:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(repo_id=RAD_REPO, filename=RAD_FILE)
        obj = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(obj, dict):                                       # file is a state_dict -> rebuild Inception-v3 and load it
            sd = obj.get('state_dict', obj.get('model', obj))
            for pre in ['module.', 'backbone.', 'model.', 'base_model.', 'features.']:
                if all(k.startswith(pre) for k in sd):
                    sd = {k[len(pre):]: v for k, v in sd.items()}
            print('RadImageNet file is a state_dict with', len(sd), 'tensors, e.g.', list(sd)[:3])
            best = None
            for builder in ['timm', 'torchvision']:
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0) if builder == 'timm' else __import__('torchvision').models.inception_v3(weights=None, aux_logits=False, init_weights=False)
                msd = cand.state_dict()
                ok = {k: v for k, v in sd.items() if k in msd and msd[k].shape == v.shape}
                isbb = lambda k: not k.startswith(('fc.', 'AuxLogits'))
                frac = sum(v.numel() for k, v in ok.items() if isbb(k)) / sum(v.numel() for k, v in msd.items() if isbb(k))
                print(f'  {builder} inception_v3: {len(ok)}/{len(msd)} tensors match ({frac:.0%} of backbone weights)')
                if best is None or frac > best[0]:
                    best = (frac, builder, cand, ok)
            frac, builder, m, ok = best
            if frac < 0.95:                                             # saved as nn.Sequential ('0.conv.weight', ...): map by position + shape
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0); msd = cand.state_dict()
                tk = [k for k in msd if not k.startswith(('fc.', 'AuxLogits'))]
                sv = list(sd.values())
                if len(tk) == len(sv) and all(msd[a].shape == v.shape for a, v in zip(tk, sv)):
                    m, ok, frac, builder = cand, dict(zip(tk, sv)), 1.0, 'timm (by position)'
                    print(f'  positional mapping: {len(tk)}/{len(tk)} tensors, all shapes match')
            assert frac > 0.95, ('RadImageNet weights do not match Inception-v3 (only %.0f%% loaded) - check the RadImageNet weight file' % (100 * frac))
            m.load_state_dict(ok, strict=False)
            print(f'  loaded into {builder} inception_v3 ({frac:.0%} of weights)')
        elif isinstance(obj, torch.jit.ScriptModule):
            m = obj
        else:
            m = obj
        for attr in ['fc', 'classifier', 'head']:                       # drop a classification head if present
            if hasattr(m, attr) and isinstance(getattr(m, attr), nn.Module):
                setattr(m, attr, nn.Identity())
        if hasattr(m, 'aux_logits'):
            m.aux_logits = False; m.AuxLogits = None
        _RAD_CACHE['m'] = m
        print('RadImageNet backbone:', type(m).__name__, '| file:', path)
    return Pooled(copy.deepcopy(_RAD_CACHE['m']))

def make_backbone(name, init):
    if init == 'RAD':
        return radimagenet_backbone()
    bb = timm.create_model(name, pretrained=PRETRAINED and init != 'SCRATCH', num_classes=0)
    if init == 'IN':
        return bb
    f = (f'{BSD}/model_bs80k_{name}.pt' if init == 'BS' else
         f'{OUT}/models/model_bs80k_{name}_frac{init[3:]}.pt' if init.startswith('BSf') else       # 'BSf0.10' (independent subsets)
         f'{OUT}/models/model_bs80k_{name}_nested{init[3:]}.pt')                                     # 'BSn0.10' (nested subsets, equal steps)
    sd = torch.load(f, map_location='cpu')
    res = bb.load_state_dict({k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}, strict=False)
    assert not res.missing_keys, f'missing keys in {f}: {res.missing_keys[:3]}'
    return bb

class DualViewNet(nn.Module):
    def __init__(self, name, n, init):
        super().__init__()
        self.backbone = make_backbone(name, init)
        self.backbone.eval()
        with torch.no_grad():
            c = self.backbone(torch.zeros(1, 3, H, W)).shape[1]
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * c, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n))
    def features(self, a, p):
        return torch.cat([self.backbone(a), self.backbone(p)], 1)
    def forward(self, a, p):
        return self.head(self.features(a, p))

def predict(model, idx, amp, feats=False):
    model.eval(); P, Fz = [], []
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp):
        for i in range(0, len(idx), 32):
            a, p = to_input(idx[i:i + 32], False)
            z = model.features(a, p); P.append(F.softmax(model.head(z).float(), 1).cpu())
            if feats:
                Fz.append(z.float().cpu())
    P = torch.cat(P).numpy() if P else np.zeros((0, 1))
    return (P, torch.cat(Fz).numpy().astype(np.float32)) if feats else P

def macro_f1(pred, y, n):
    f = []
    for c in range(n):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def train(name, init, n, tr_idx, tr_y, va_idx, va_y, seed, epochs=EPOCHS, patience=PATIENCE, amp=True):
    set_seed(seed); amp = amp and DEVICE == 'cuda'
    model = DualViewNet(name, n, init).to(DEVICE)
    cnt = np.bincount(tr_y, minlength=n)
    w = torch.tensor(len(tr_y) / (n * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-4}, {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=amp)
    rng = np.random.default_rng(seed); yt = torch.tensor(tr_y, device=DEVICE)
    best, state, wait, best_ep, ep = -1.0, None, 0, 0, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = rng.permutation(len(tr_idx)); nb = max(len(perm) // BATCH, 1)
        for b in range(nb):
            j = perm[b * BATCH:(b + 1) * BATCH]
            a, p = to_input(tr_idx[j], True, rng)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                loss = loss_fn(model(a, p), yt[j])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va_idx, amp)
        vf = macro_f1(pv.argmax(1), va_y, n) if not np.isnan(pv).any() else -1
        if vf > best:
            best, state, wait, best_ep = vf, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    return model, dict(best_inner_f1=best, best_epoch=best_ep, epochs_run=ep, amp=amp)

def ids_to_idx(ids):
    return np.array([IDX[p] for p in ids])

def save_preds(path, P, ids, y=None):
    d = pd.DataFrame(P, columns=[f'prob_{i}' for i in range(P.shape[1])])
    if y is not None:
        d.insert(0, 'y_true', y)
    d.insert(0, 'patient_id', list(ids)); d.insert(1, 'camera', [CAM[p] for p in ids]); d.to_csv(path, index=False)

def done_keys():
    return {f[:-5] for f in os.listdir(f'{OUT}/logs') if f.endswith('.json')}

def log_run(row):
    # one small JSON per finished run (safe if Colab disconnects); runs.csv is rebuilt from all of them
    with open(f"{OUT}/logs/{row['key']}.json", 'w') as fh:
        json.dump({k: (v.item() if hasattr(v, 'item') else v) for k, v in row.items()}, fh)
    rows = [json.load(open(f'{OUT}/logs/{f}')) for f in sorted(os.listdir(f'{OUT}/logs')) if f.endswith('.json')]
    pd.DataFrame(rows).to_csv(RUNS, index=False)

def time_left():
    return MAX_HOURS - (time.time() - T_START) / 3600

def run_job(key, exp, name, init, n, seed, tr, va, evals, feat_sets=(), extra=None):
    """tr/va: DataFrames with patient_id and y. evals: {kind: (ids, y or None)}. Saves preds of every eval set."""
    t0 = time.time()
    for amp in [True, False]:
        model, info = train(name, init, n, ids_to_idx(tr.patient_id), tr.y.values, ids_to_idx(va.patient_id), va.y.values, seed, amp=amp)
        outs, bad = {}, False
        for kind, (ids, y) in evals.items():
            res = predict(model, ids_to_idx(ids), info['amp'], feats=kind in feat_sets)
            P = res[0] if kind in feat_sets else res
            bad |= bool(np.isnan(P).any()); outs[kind] = res
        if not bad or not amp or DEVICE != 'cuda':
            break
        print('  NaN -> retrying in full precision'); del model; gc.collect(); torch.cuda.empty_cache()
    for kind, (ids, y) in evals.items():
        res = outs[kind]; P = res[0] if kind in feat_sets else res
        save_preds(f'{OUT}/preds/{key}_{kind}.csv', P, ids, y)
        if kind in feat_sets:
            np.savez_compressed(f'{OUT}/feats/{key}_{kind}.npz', ids=np.array(ids), z=np.clip(res[1], -6e4, 6e4).astype(np.float16))
    row = dict(key=key, exp=exp, model=name, init=init, seed=seed, minutes=round((time.time() - t0) / 60, 2), n_train=len(tr), n_inner=len(va), **info, **(extra or {}))
    for kind, (ids, y) in evals.items():
        if y is not None:
            res = outs[kind]; P = res[0] if kind in feat_sets else res
            row[f'{kind}_f1'] = macro_f1(P.argmax(1), np.asarray(y), n)
    log_run(row)
    print(f"  {key}: inner {info['best_inner_f1']:.3f} | " + ' | '.join(f"{k[:-3]} {v:.3f}" for k, v in row.items() if k.endswith('_f1') and k != 'best_inner_f1') + f" | {row['minutes']} min", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

def inner_split(df, seed=0):
    for strat in [df.label + '_' + df.machine, df.label, None]:     # class x camera; fall back if a stratum is too small
        try:
            tr, va = train_test_split(df, test_size=INNER_FRAC, stratify=strat, random_state=seed)
            return tr.reset_index(drop=True), va.reset_index(drop=True)
        except ValueError:
            continue

def eta(exp, remaining):
    if not os.path.exists(RUNS):
        return ''
    r = pd.read_csv(RUNS); r = r[r.exp == exp] if 'exp' in r else r.iloc[:0]
    return f'(~{remaining * r.minutes.mean() / 60:.1f} h left for {exp})' if len(r) else ''
print('Ready')

In [ ]:
# domain adaptation (DANN, CORAL) - same code as notebook 05
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None

def coral_loss(hs, ht):
    """CORAL penalty as implemented in DomainBed (Gulrajani & Lopez-Paz, 2021): squared difference of feature means + covariances"""
    hs, ht = hs.float(), ht.float()
    cs, ct = torch.cov(hs.T), torch.cov(ht.T)
    return (hs.mean(0) - ht.mean(0)).pow(2).mean() + (cs - ct).pow(2).mean()

def train_da(method, name, init, n, tr_idx, tr_y, va_idx, va_y, tg_idx, seed, epochs=EPOCHS, patience=PATIENCE, amp=True):
    """same protocol as train() (notebook 03) + unlabelled target images; model selection on the source inner split only"""
    set_seed(seed); amp = amp and DEVICE == 'cuda'
    model = DualViewNet(name, n, init).to(DEVICE)
    c2 = model.head[1].in_features
    dom = nn.Sequential(nn.Linear(c2, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1)).to(DEVICE)
    cnt = np.bincount(tr_y, minlength=n)
    w = torch.tensor(len(tr_y) / (n * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    groups = [{'params': model.backbone.parameters(), 'lr': 1e-4}, {'params': model.head.parameters(), 'lr': 1e-3}]
    if method == 'dann':
        groups.append({'params': dom.parameters(), 'lr': 1e-3})
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=amp)
    rng = np.random.default_rng(seed); trng = np.random.default_rng(seed + 7919); yt = torch.tensor(tr_y, device=DEVICE)
    nb = max(len(tr_idx) // BATCH, 1); total = epochs * nb; step = 0
    coral_ref = None   # CORAL weight: penalty divided by its value at the first training step (train mode, unlabelled data only),
                       # so the CORAL term starts at DA_WEIGHT, comparable to the classification loss (weighting rule of Sun & Saenko 2016)
    best, state, wait, best_ep, ep, da_hist = -1.0, None, 0, 0, 0, []
    for ep in range(1, epochs + 1):
        model.train(); dom.train(); ep_da = []
        perm = rng.permutation(len(tr_idx))
        for b in range(nb):
            j = perm[b * BATCH:(b + 1) * BATCH]
            tj = trng.choice(len(tg_idx), size=BATCH, replace=len(tg_idx) < BATCH)
            a_s, p_s = to_input(tr_idx[j], True, rng); a_t, p_t = to_input(tg_idx[tj], True, rng)
            ns = len(j); lam = 2.0 / (1.0 + np.exp(-10.0 * step / total)) - 1.0; step += 1
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                z = model.features(torch.cat([a_s, a_t]), torch.cat([p_s, p_t]))
                h = model.head[:3](z)
                cls = loss_fn(model.head[3:](h[:ns]), yt[j])
                if method == 'dann':
                    dlab = torch.cat([torch.zeros(ns, device=DEVICE), torch.ones(len(tj), device=DEVICE)])
                    dl = F.binary_cross_entropy_with_logits(dom(GradReverse.apply(z, lam)).squeeze(1).float(), dlab)
                elif method == 'coralstd':                   # standard CORAL penalty (DomainBed), weight 1, not rescaled
                    dl = coral_loss(h[:ns], h[ns:])
                else:                                        # 'coral': rescaled variant (first run; over-aligned, kept for reproducibility)
                    dl = coral_loss(h[:ns], h[ns:])
                    if coral_ref is None:
                        v0 = float(dl.detach()); coral_ref = v0 if np.isfinite(v0) and v0 > 1e-8 else 1.0
                    dl = dl / coral_ref
                loss = cls + DA_WEIGHT[method] * dl
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); ep_da.append(float(dl.detach()))
        sched.step(); da_hist.append(float(np.mean(ep_da)))
        pv = predict(model, va_idx, amp)
        vf = macro_f1(pv.argmax(1), va_y, n) if not np.isnan(pv).any() else -1
        if vf > best:
            best, state, wait, best_ep = vf, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    del dom
    return model, dict(best_inner_f1=best, best_epoch=best_ep, epochs_run=ep, amp=amp, coral_ref=coral_ref if (method == 'coral' and coral_ref is not None) else np.nan,
                       da_loss_first=da_hist[0] if da_hist else np.nan, da_loss_best=da_hist[best_ep - 1] if best_ep else np.nan)

In [ ]:
# intensity augmentation and MixStyle - same code as notebook 06
CUR = {'method': None}                           # method of the run in progress (read by to_input and DualViewNet below)

def inorm(x):
    """per-image percentile rescaling (0.5th -> 0, 99.5th -> 1); x: B x 1 x H x W in [0,1]"""
    q = torch.quantile(x.flatten(1), torch.tensor([0.005, 0.995], device=x.device), dim=1)    # 2 x B
    lo, hi = q[0].view(-1, 1, 1, 1), q[1].view(-1, 1, 1, 1)
    return ((x - lo) / (hi - lo).clamp(min=1e-3)).clamp(0, 1)

def iaug(img, rng):
    """scanner-aware intensity augmentation (after the standard augmentation), per image: gamma, blur, noise - each p = 0.5"""
    out = []
    for x in img:                                                    # x: 1 x H x W in [0,1]
        if rng.random() < 0.5:
            x = x.clamp(1e-6, 1) ** float(rng.uniform(0.7, 1.4))
        if rng.random() < 0.5:
            x = TF.gaussian_blur(x, kernel_size=7, sigma=float(rng.uniform(0.3, 1.5)))
        if rng.random() < 0.5:
            g = torch.Generator(device=x.device).manual_seed(int(rng.integers(2**31)))
            x = (x + torch.randn(x.shape, generator=g, device=x.device) * float(rng.uniform(0.005, 0.03))).clamp(0, 1)
        out.append(x)
    return torch.stack(out)

def to_input(idx, train, rng=None):              # replaces the shared version; identical when CUR['method'] is None or 'mixstyle'
    x = X[idx].float() / 255.0                                       # B x 2 x H x W
    a, p = x[:, :1], x[:, 1:]
    if CUR['method'] == 'inorm':
        a, p = inorm(a), inorm(p)
    if train:
        a, p = augment(a, rng), augment(p, rng)
        if CUR['method'] == 'iaug':
            a, p = iaug(a, rng), iaug(p, rng)
    norm = lambda t: (t.repeat(1, 3, 1, 1) - MEAN) / STD
    return norm(a), norm(p)

MIX_LAYERS = {'tf_efficientnetv2_s': ['blocks.0', 'blocks.1', 'blocks.2'], 'resnext50_32x4d': ['layer1', 'layer2', 'layer3'],
              'regnety_016': ['s1', 's2', 's3'], 'inception_v3': ['Mixed_5b', 'Mixed_5d', 'Mixed_6e']}
MIX_P, MIX_ALPHA = 0.5, 0.1
_beta = torch.distributions.Beta(torch.tensor(MIX_ALPHA), torch.tensor(MIX_ALPHA))

def mixstyle_hook(module, inp, x):
    """MixStyle (Zhou et al., 2021), random shuffle within the batch; active only in training mode"""
    if not module.training or x.dim() != 4 or x.size(0) < 2 or random.random() > MIX_P:
        return None
    B = x.size(0); xf = x.float()
    mu = xf.mean(dim=[2, 3], keepdim=True); sig = (xf.var(dim=[2, 3], keepdim=True) + 1e-6).sqrt()
    mu, sig = mu.detach(), sig.detach()
    lmda = _beta.sample((B, 1, 1, 1)).to(x.device); perm = torch.randperm(B, device=x.device)
    mu2, sig2 = mu * lmda + mu[perm] * (1 - lmda), sig * lmda + sig[perm] * (1 - lmda)
    return (((xf - mu) / sig) * sig2 + mu2).to(x.dtype)

_DualViewNetBase = DualViewNet
class DualViewNet(_DualViewNetBase):             # replaces the shared version; adds MixStyle hooks only for method 'mixstyle'
    def __init__(self, name, n, init):
        super().__init__(name, n, init)
        self.n_mix = 0
        if CUR['method'] == 'mixstyle':
            for ln in MIX_LAYERS[name]:
                self.backbone.get_submodule(ln).register_forward_hook(mixstyle_hook); self.n_mix += 1

In [ ]:
# training: BS-80K source, every method, every backbone and seed
FOLD = dict(zip(sp.patient_id, sp.fold))
LABEL = dict(zip(sp.patient_id, sp.label))
BS_TR = bsplit[bsplit.split == 'train'][['patient_id', 'y']].reset_index(drop=True)
BS_VA = bsplit[bsplit.split == 'val'][['patient_id', 'y']].reset_index(drop=True)
BS_TE = bsplit[bsplit.split == 'test'][['patient_id', 'y']].reset_index(drop=True)
TGT = {c: sp[sp.machine == c].assign(y=lambda d: (d.label == 'SECONDARY').astype(int))[['patient_id', 'y']].reset_index(drop=True) for c in CAMERAS}
UNL = {c: ind[ind.machine == c].patient_id.tolist() for c in CAMERAS}

def jobs():
    J = []
    for seed in SEEDS:                           # seed 42 for everything first
        for method in METHODS:
            for name in MODELS:
                for tgt in (CAMERAS if method in DA_METHODS else [None]):
                    J.append((f'p19_{method}_{name}_s{seed}' + (f'_to{tgt}' if tgt else ''), method, name, seed, tgt))
    return J
JOBS = jobs()
print('runs planned:', len(JOBS), '| per seed:', len(JOBS) // len(SEEDS))

def run_one(key, method, name, seed, tgt):
    t0 = time.time()
    evals = {'bstest': (BS_TE.patient_id, BS_TE.y.values)}
    for c in ([tgt] if tgt else CAMERAS):
        evals[c] = (TGT[c].patient_id, TGT[c].y.values)
    CUR['method'] = method if method in ('iaug', 'mixstyle') else None
    try:
        for amp in [True, False]:
            if method in DA_METHODS:
                model, info = train_da(method, name, 'IN', 2, ids_to_idx(BS_TR.patient_id), BS_TR.y.values, ids_to_idx(BS_VA.patient_id),
                                       BS_VA.y.values, ids_to_idx(UNL[tgt]), seed, amp=amp)
            else:
                model, info = train(name, 'IN', 2, ids_to_idx(BS_TR.patient_id), BS_TR.y.values, ids_to_idx(BS_VA.patient_id), BS_VA.y.values, seed, amp=amp)
            outs = {k: predict(model, ids_to_idx(ids), info['amp']) for k, (ids, y) in evals.items()}
            if not any(np.isnan(P).any() for P in outs.values()) or not amp or DEVICE != 'cuda':
                break
            print('  NaN -> retrying in full precision'); del model; gc.collect(); torch.cuda.empty_cache()
    finally:
        CUR['method'] = None
    for k, (ids, y) in evals.items():
        save_preds(f'{OUT}/preds/{key}_{k}.csv', outs[k], ids, y)
    row = dict(key=key, exp='bs80k_to_machines', method=method, model=name, seed=seed, target=tgt or 'both',
               minutes=round((time.time() - t0) / 60, 2), n_train=len(BS_TR), n_val=len(BS_VA),
               n_unlabelled=len(UNL[tgt]) if tgt else 0, **{k: v for k, v in info.items() if np.isscalar(v)})
    for k, (ids, y) in evals.items():
        row[f'{k}_f1'] = macro_f1(outs[k].argmax(1), np.asarray(y), 2)
    log_run(row)
    print(f"  {key}: BS-80K val {info['best_inner_f1']:.3f} | " + ' | '.join(f"{k} {row[k + '_f1']:.3f}" for k in evals) +
          f" | {row['minutes']} min", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

def run_all():
    done = done_keys(); todo = [j for j in JOBS if j[0] not in done]
    print(f'{len(JOBS) - len(todo)}/{len(JOBS)} done', eta('bs80k_to_machines', len(todo)))
    for key, method, name, seed, tgt in todo:
        if time_left() <= 0:
            print('MAX_HOURS reached - restart the runtime and run all again to resume'); return False
        print(f'[{method}] {name} | seed {seed}' + (f' | unlabelled target {tgt} ({len(UNL[tgt])} primary patients)' if tgt else ''))
        run_one(key, method, name, seed, tgt)
    return True

OK = run_all() if RUN_TRAINING else True
if os.path.exists(RUNS):
    r = pd.read_csv(RUNS)
    print(r.groupby('method').agg(runs=('key', 'count'), minutes=('minutes', 'mean'), bs80k_test_f1=('bstest_f1', 'mean')).round(3))
print('ALL RUNS FINISHED' if OK else 'NOT FINISHED YET - Run all again (finished runs are skipped); the analysis below uses what is done')

In [ ]:
# Analysis 1: seed-averaged predictions per method, backbone and target machine; recalibration
from functools import lru_cache
@lru_cache(maxsize=None)
def _rd(path):
    return pd.read_csv(path) if os.path.exists(path) else None
def rd(path):
    x = _rd(path); return None if x is None else x.copy()

def pred_file(method, name, seed, tgt):
    m = 'none' if method == 'recal' else method
    return f'{OUT}/preds/p19_{m}_{name}_s{seed}' + (f'_to{tgt}' if m in DA_METHODS else '') + f'_{tgt}.csv'

def seeds_done(method, name, tgt):
    return [s for s in SEEDS if os.path.exists(pred_file(method, name, s, tgt))]

def avg(lst):
    out = lst[0][['patient_id', 'camera', 'y_true']].copy()
    for x in lst:
        assert (x.patient_id.values == out.patient_id.values).all()
    out['p1'] = np.mean([x.prob_1.values for x in lst], 0); return out

def best_shift(p, y):
    """log-odds shift maximising macro-F1 on the calibration patients (grid -4..4, step 0.1); ties -> smallest shift"""
    L = np.log(np.clip(p, 1e-7, 1 - 1e-7) / np.clip(1 - p, 1e-7, 1)); best, bs = -1, 0.0
    for s in sorted(np.round(np.arange(-4, 4.001, 0.1), 2), key=abs):
        f = macro_f1((L + s > 0).astype(int), y, 2)
        if f > best + 1e-12:
            best, bs = f, s
    return bs

def recal(df, task_ids, rep_seed):
    """threshold of the 'none' model set on 20 labelled patients of the same machine and task, from the other folds"""
    d = df[df.patient_id.isin(task_ids)].sort_values('patient_id').reset_index(drop=True); out = []
    L = np.log(np.clip(d.p1.values, 1e-7, 1 - 1e-7) / np.clip(1 - d.p1.values, 1e-7, 1))
    fk = d.patient_id.map(FOLD).values
    for k in QUICK_FOLDS:
        pool = d[fk != k]; n_cal = min(N_RECAL, len(pool) - 2)
        try:
            cal, _ = train_test_split(pool, train_size=n_cal, stratify=pool.y_true, random_state=rep_seed + k)
        except ValueError:
            cal, _ = train_test_split(pool, train_size=n_cal, random_state=rep_seed + k)
        s = best_shift(cal.p1.values, cal.y_true.values.astype(int))
        out.append(pd.Series((L[fk == k] + s > 0).astype(int), index=d.index[fk == k]))
    return pd.concat(out).sort_index().values

TASK_IDS = {(c, t): sp[(sp.machine == c) & sp.label.isin(TASKS[t])].patient_id.sort_values().tolist() for c in CAMERAS for t in TASKS}
def e12_probs(task, name, init, train_cam, test_cam):
    """Experiment 1 (train_cam == test_cam) or Experiment 2 predictions for the patients of test_cam, seed-averaged"""
    L = []
    for s in SEEDS:
        F_ = []
        for k in QUICK_FOLDS:
            kind = 'oof' if train_cam == test_cam else 'cross'
            x = rd(f'{EXP12}/preds/mach_{E12_TASK[task]}_{name}_{init}_{train_cam}_s{s}_fold{k}_{kind}.csv')
            if x is None:
                return None
            F_.append(x[x.patient_id.map(FOLD) == k])
        L.append(pd.concat(F_).sort_values('patient_id').reset_index(drop=True))
    out = L[0][['patient_id', 'y_true']].copy(); out['p1'] = np.mean([x.prob_1.values for x in L], 0); return out

PRED = {}       # (arm, name, tgt, task) -> DataFrame patient_id, y_true, pred (0/1), p1
for name in MODELS:
    for c in CAMERAS:
        for method in METHODS:
            ss = seeds_done(method, name, c)
            if not ss:
                continue
            df = avg([rd(pred_file(method, name, s, c)) for s in ss])
            for t in TASKS:
                d = df[df.patient_id.isin(TASK_IDS[(c, t)])].sort_values('patient_id').reset_index(drop=True)
                PRED[(method, name, c, t)] = d.assign(pred=(d.p1 > 0.5).astype(int), n_seeds=len(ss))
                if method == 'none':
                    reps = np.stack([recal(df, TASK_IDS[(c, t)], 5000 + 100 * r) for r in range(RECAL_REPS)])
                    PRED[('recal', name, c, t)] = d.assign(pred_reps=list(reps.T), n_seeds=len(ss))
        for t in TASKS:                          # references from Experiments 1 and 2 (ImageNet start, same 3 seeds)
            for arm, src in [('own_machine', c), ('other_machine', 'M2' if c == 'M1' else 'M1')]:
                e = e12_probs(t, name, 'IN', src, c)
                if e is not None:
                    e = e[e.patient_id.isin(TASK_IDS[(c, t)])].sort_values('patient_id').reset_index(drop=True)
                    PRED[(arm, name, c, t)] = e.assign(pred=(e.p1 > 0.5).astype(int))
print('prediction sets:', len(PRED))

In [ ]:
# Analysis 2: macro-F1, AUC, sensitivity, specificity + joint patient bootstrap vs 'none'
from sklearn.metrics import roc_auc_score
cam_patients = {c: sorted(sp[sp.machine == c].patient_id) for c in CAMERAS}
cam_pos = {c: {p: i for i, p in enumerate(cam_patients[c])} for c in CAMERAS}
rng = np.random.default_rng(2026)
BOOT = {c: np.stack([np.bincount(rng.integers(0, len(cam_patients[c]), len(cam_patients[c])), minlength=len(cam_patients[c]))
                     for _ in range(N_BOOT)]).astype(np.float32) for c in CAMERAS}

def wf1(pred, y, ids, cam):
    Wm = BOOT[cam][:, [cam_pos[cam][p] for p in ids]]; f = []
    for j in (0, 1):
        tp = Wm @ ((pred == j) & (y == j)).astype(np.float32); fp = Wm @ ((pred == j) & (y != j)).astype(np.float32)
        fn = Wm @ ((pred != j) & (y == j)).astype(np.float32)
        f.append(np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0))
    return np.mean(f, 0)

def pval(d):
    return min(1.0, 2 * min((np.sum(d <= 0) + 1) / (len(d) + 1), (np.sum(d >= 0) + 1) / (len(d) + 1)))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r_, i in enumerate(o):
        run = max(run, (m - r_) * p[i]); adj[i] = min(1.0, run)
    return adj

ARMS = ['none', 'recal', 'iaug', 'mixstyle', 'dann', 'coralstd', 'own_machine', 'other_machine']
ROWS, DIST = [], {}
for (arm, name, c, t), d in PRED.items():
    y = d.y_true.values.astype(int); ids = d.patient_id.tolist()
    if arm == 'recal':
        reps = np.stack(d.pred_reps.values).T
        f1 = float(np.mean([macro_f1(r, y, 2) for r in reps])); sens = float(np.mean([np.mean(r[y == 1] == 1) for r in reps]))
        spec = float(np.mean([np.mean(r[y == 0] == 0) for r in reps])); bs_ = np.mean([wf1(r, y, ids, c) for r in reps], 0); auc = np.nan
    else:
        pr = d.pred.values; f1 = macro_f1(pr, y, 2); sens = float(np.mean(pr[y == 1] == 1)); spec = float(np.mean(pr[y == 0] == 0))
        bs_ = wf1(pr, y, ids, c)
        try:
            auc = roc_auc_score(y, d.p1.values)
        except ValueError:
            auc = np.nan
    DIST[(arm, name, c, t)] = bs_
    ROWS.append(dict(arm=arm, model=MN[name], name=name, target=c, task=t, n=len(y), f1=f1, auc=auc, sens=sens, spec=spec,
                     lo=np.percentile(bs_, 2.5), hi=np.percentile(bs_, 97.5)))
A = pd.DataFrame(ROWS)
for i, r in A.iterrows():
    base = DIST.get(('none', r['name'], r.target, r.task))
    if base is not None and r.arm != 'none':
        d_ = DIST[(r.arm, r['name'], r.target, r.task)] - base
        A.loc[i, 'minus_none'] = r.f1 - A[(A.arm == 'none') & (A.name == r['name']) & (A.target == r.target) & (A.task == r.task)].f1.iloc[0]
        A.loc[i, 'minus_none_p'] = pval(d_)
if 'minus_none_p' in A:
    A['minus_none_pholm'] = np.nan
    for arm, g in A[A.minus_none_p.notna()].groupby('arm'):          # family: backbones x tasks x target machines
        A.loc[g.index, 'minus_none_pholm'] = holm(g.minus_none_p.values)
A = A.sort_values(['task', 'target', 'arm', 'model']).reset_index(drop=True)
A.drop(columns=['name']).to_csv(f'{OUT}/analysis/per_backbone.csv', index=False)

In [ ]:
# Analysis 3: pooled over backbones, per task and target machine (and both machines)
P_ROWS = []
for t in TASKS:
    for scope in CAMERAS + ['both']:
        cams = CAMERAS if scope == 'both' else [scope]
        for arm in ARMS:
            keys = [(arm, n, c, t) for n in MODELS for c in cams if (arm, n, c, t) in DIST]
            if not keys:
                continue
            sub = A[(A.arm == arm) & (A.task == t) & A.target.isin(cams)]
            row = dict(task=t, target=scope, arm=arm, n_pairs=len(keys), f1=sub.f1.mean(), auc=sub.auc.mean(), sens=sub.sens.mean(), spec=sub.spec.mean())
            bk = [('none', k[1], k[2], t) for k in keys]
            if arm != 'none' and all(b in DIST for b in bk):
                d_ = np.mean([DIST[k] for k in keys], 0) - np.mean([DIST[b] for b in bk], 0)
                base = A[(A.arm == 'none') & (A.task == t) & A.target.isin(cams)].f1.mean()
                row.update(minus_none=row['f1'] - base, lo=np.percentile(d_, 2.5), hi=np.percentile(d_, 97.5), p=pval(d_),
                           n_better=int((sub.minus_none > 0).sum()))
            P_ROWS.append(row)
PO = pd.DataFrame(P_ROWS); PO.to_csv(f'{OUT}/analysis/pooled.csv', index=False)
lines = [f'BS-80K -> Machine 1 / Machine 2 | backbones {len(MODELS)} | bootstrap {N_BOOT}']
for _, r in PO.iterrows():
    s = f"{r.task} | {r.target} | {r.arm} | F1 {r.f1:.3f} AUC {r.auc:.3f} sens {r.sens:.3f} spec {r.spec:.3f}"
    if not pd.isna(r.get('minus_none', np.nan)):
        s += f" | vs none {r.minus_none:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}] p {r.p:.4f} | better {int(r.n_better)}/{int(r.n_pairs)}"
    lines.append(s)
open(f'{OUT}/analysis/summary.txt', 'w').write('\n'.join(lines)); print('\n'.join(lines))
print('Saved:', f'{OUT}/analysis/summary.txt')